# 🧪 Vigyan-1.5B 4x-MoE: Compact Edge Mixture of Experts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/vigyan_1_5b_moe_demo.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Vigyan-1.5B-4x-MoE-yellow)](https://huggingface.co/shreyansh12183/Vigyan-1.5B-4x-MoE)

---

### 🧪 Architectural Overview
**Vigyan-1.5B 4× MoE** is a compact Sparse Mixture of Experts model upcycled from 4 specialized domain adapters (Science, Technology, Engineering, Mathematics) on `DeepSeek-R1-Distill-Qwen-1.5B`.

* **Sparse Routing:** Top-1 active expert gating per token with contrasting negative prompt router calibration.
* **Footprint:** Consumes < 1.5 GB VRAM in 4-bit NF4; standalone Q4_K_M GGUF available for instant mobile/CPU deployment.

> **License:** Creative Commons Attribution-NonCommercial 4.0 International (**CC BY-NC 4.0**). Free for non-commercial academic research and student study.

### Step 1: Install Dependencies (Runs in ~30s on free Colab T4)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes gradio
print('✅ Environment ready.')

### Step 2: Load Model in 4-bit NF4 Precision (Consumes < 3 GB VRAM)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'shreyansh12183/Vigyan-1.5B-4x-MoE'
print(f'Loading 1.5B 4x-MoE Model: {MODEL_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)
print('✅ 1.5B 4x-MoE Model loaded in ~1.4 GB VRAM!')


### Step 3: Launch Interactive Gradio WebUI

In [ ]:
import gradio as gr

def generate_response(prompt, temperature, max_tokens):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 1e-2),
            do_sample=temperature > 0.05,
            pad_token_id=tokenizer.eos_token_id
        )
    gen = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return gen

demo = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(lines=4, label='Scientific / STEM Query', placeholder='Enter your problem...'),
        gr.Slider(0.0, 1.0, value=0.2, step=0.05, label='Temperature'),
        gr.Slider(64, 1024, value=512, step=64, label='Max Tokens')
    ],
    outputs=gr.Textbox(lines=10, label='Model Output & Mathematical Derivation'),
    title='🧪 Vigyan-1.5B 4x-MoE: Compact Edge Mixture of Experts',
    description='Non-commercial research demonstration powered by Vigyan AI Sovereign Infrastructure under CC BY-NC 4.0.',
    examples=[['Explain the difference between Sallen-Key low-pass filter and multiple-feedback active filter topologies.'], ['Evaluate the definite integral of sin^3(x) / (sin^3(x) + cos^3(x)) from 0 to pi/2.'], ['How does crystal field splitting explain why [Fe(H2O)6]2+ is paramagnetic with 4 unpaired electrons?']]
)

demo.launch(share=True, debug=False)
